In [1]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import numpy as np
import pandas as pd
from pandarallel import pandarallel
pandarallel.initialize()
from functools import reduce
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.DataStructs import TanimotoSimilarity
from rdkit.Chem.AllChem import GetMorganFingerprintAsBitVect
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default_config = {
    "toImageButtonOptions": {"format": "png", "scale": 5}}

# sys.path.append(os.path.abspath(".."))
# from new_geass import Geass
# from AIDDTools import fingerprint_tools as fpt

sys.path.append(os.path.abspath(".."))
import utils

clear_output()

In [2]:
# Compile the Metabolic Unit Database (MUDB)
df_kegg = pd.read_parquet('/home/mattia/geass/data/Database_KEGG_cpd.gzip', engine='fastparquet')
df_kegg = df_kegg.dropna(subset='ISOSMILES')
PandasTools.AddMoleculeColumnToFrame(df_kegg, 'ISOSMILES')
df_kegg = df_kegg.dropna(subset='ROMol')

df_kegg = df_kegg.rename(columns={'ENTRY':'ID', 'MOL_WEIGHT':'MW'})
df_kegg = df_kegg.rename(columns={'BRITE':'Ontology'})
df_kegg = df_kegg[['ID', 'ISOSMILES', 'MW', 'Ontology', 'ROMol']]

df_LM = pd.read_pickle('/home/mattia/geass/data/Database_lipidmaps.pkl')
df_LM = df_LM.rename(columns={'FORMULA':'Formula', 'ONTOLOGY':'Ontology'})
df_LM['Ontology'] = df_LM['Ontology'].parallel_apply(lambda x: '; '.join(x) if isinstance(x, list) else x)

In [ ]:
df_MU = pd.concat([df_kegg, df_LM], ignore_index=True)
df_MU = df_MU.dropna(subset=["MW", "ROMol"]).reset_index(drop=True)
df_MU['ROMol'] = df_MU['ROMol'].parallel_apply(utils.standardize_mol)
df_MU['ISOSMILES'] = df_MU['ROMol'].parallel_apply(lambda x: Chem.MolToSmiles(x))
df_MU = df_MU.drop_duplicates(subset='ISOSMILES').reset_index(drop=True)

allowed_elements = {"H", "C", "N", "O", "F", "P", "S", "Cl", "Br", "I"}
df_MU = df_MU[df_MU["ROMol"].apply(lambda mol: all(atom.GetSymbol() in allowed_elements for atom in mol.GetAtoms()))].reset_index(drop=True)

df_MU['InChIKey'] = df_MU['ROMol'].parallel_apply(Chem.MolToInchiKey)
df_MU['Formula'] = df_MU['ROMol'].parallel_apply(lambda x: AllChem.CalcMolFormula(x))
df_MU["MW"] = df_MU["MW"].round(3)
df_MU["Num_Atoms"] = df_MU["ROMol"].parallel_apply(lambda x: x.GetNumHeavyAtoms())
# df_MU["Feature_vec"] = df_MU["ROMol"].parallel_apply(calc_feature_vec)

# Calculate fingerprints
sys.path.append('/home/mattia/aiddtools')
import fingerprint_tools as fpt
df_MU['FP_atompair'] = df_MU['ROMol'].parallel_map(
    lambda x: fpt.calculate_fp(x, 'atompair', as_numpy=True))
df_MU['FP_map4'] = df_MU['ROMol'].parallel_map(
    lambda x: fpt.calculate_fp(x, 'map4', as_numpy=True))

# Convert fps to contiguous NumPy arrays for efficient storage
fp_atompair = np.ascontiguousarray(np.stack(df_MU["FP_atompair"].to_numpy()))
fp_map4 = np.ascontiguousarray(np.stack(df_MU["FP_map4"].to_numpy()))

# Save fps separately in compressed NumPy format
np.savez_compressed("/home/mattia/geass/data/MUDB_FP_atompair.npz", fp=fp_atompair)
np.savez_compressed("/home/mattia/geass/data/MUDB_FP_map4.npz", fp=fp_map4)

# Save metadata only
df_meta = df_MU.drop(columns=["ROMol", "FP_atompair", "FP_map4"])
df_meta.to_csv("/home/mattia/geass/data/MUDB.csv", index=False)
df_meta.to_pickle("/home/mattia/geass/data/MUDB.pkl")

df_MU